Read the news table from lakehouse DB

In [2]:
df=spark.sql("select * from newsdata_lake_db.news_raw");
display(df)

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 422c2c4f-4bcd-4498-9ab6-d7168226de58)

Import Synapse ML

In [3]:
import synapse.ml.core
from synapse.ml.services import AnalyzeText

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 5, Finished, Available, Finished, False)

Configure Synapse ML

In [4]:
#import the model and configure input and output columns

model=(AnalyzeText().setTextCol("description")
.setKind("SentimentAnalysis")
.setOutputCol("response")
.setErrorCol("error"))

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 6, Finished, Available, Finished, False)

Apply model to data frame

In [5]:
#Apply the model to a dataframe
result=model.transform(df)
display(result)

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5b83a3a-01e5-49c9-a1c2-3f6e1776284b)

Cross-checking the schema

In [6]:
result.printSchema()

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 8, Finished, Available, Finished, False)

root
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- link: string (nullable = true)
 |-- source: string (nullable = true)
 |-- country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- authors: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- published_date: string (nullable = true)
 |-- published_ts: timestamp (nullable = true)
 |-- published_date_formatted: string (nullable = true)
 |-- error: struct (nullable = true)
 |    |-- response: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- protocolVersion: struct (nullable = true)
 |    |    |    |-- protocol: string (nullable = true)
 |    |    |    |-- major: integer (nullable = false)
 |    |    |    |-- minor: integer (nullable = false)
 |    |    |-- statusCode: integer (nullable = false)
 |    |    |-- reasonPhrase: string (null

Addition of sentiment column into the dataframe

In [7]:
from pyspark.sql.functions import col

sentiment_df = result.withColumn(
    "sentiment",
    col("response.documents.sentiment")
)

display(sentiment_df)

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 9, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 3b3e3101-8486-443a-9295-8ca92809e05c)

Removing unnecessary columns-error and response

In [8]:
sentiment_df_final=sentiment_df.drop("error","response")
display(sentiment_df_final)

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 10, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, f87f0ee5-4116-46f8-877a-94b0bf528ec3)

sentiment_df_final->Delta format->newsdata_lake_db.tbl_sentiment_analysis-type 1 merge into the table

In [9]:
from pyspark.sql.utils import AnalysisException

try:
    table_name = "newsdata_lake_db.tbl_sentiment_analysis"

    sentiment_df_final.write.format("delta").saveAsTable(table_name)

except AnalysisException:

    print("Table Already Exists")

    sentiment_df_final.createOrReplaceTempView("vw_sentiment_df_final")

    spark.sql(f"""
        MERGE INTO {table_name} target_table
        USING vw_sentiment_df_final source_view

        ON source_view.url = target_table.url

        WHEN MATCHED AND
            source_view.title <> target_table.title OR
            source_view.description <> target_table.description OR
            source_view.category <> target_table.category OR
            source_view.image <> target_table.image OR
            source_view.provider <> target_table.provider OR
            source_view.datepublished <> target_table.datepublished

        THEN UPDATE SET *

        WHEN NOT MATCHED THEN INSERT *
    """)

StatementMeta(, cad8803f-d67d-45fe-9782-ef5401868ce1, 11, Submitted, Running, Running, True)

Table Already Exists
